In [0]:
%run "/Workspace/Users/dnp50022@gmail.com/Retail-Sales-Data-Pipeline/databricks/notebooks/silver/service principle"

In [0]:

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import (col, trim, to_date, row_number, current_timestamp, lit,concat_ws,coalesce)

# COMMAND ----------


storage_account = "salesstorageproject"
container_name = "sales"
table_name = "inventory"

bronze_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze data/{table_name}/*"


In [0]:

# Read Transaction CSV from Bronze
df = spark.read.format("csv").option("header", "true").load(bronze_path)  
display(df)

In [0]:
df.columns

In [0]:
# CAST DATATYPES
df = (
    df.withColumn("inventory_id", col("inventory_id").cast("string"))
      .withColumn("product_id", col("product_id").cast("string"))
      .withColumn("store_id", col("store_id").cast("string"))
      .withColumn("stock_quantity", col("stock_quantity").cast("int"))
      .withColumn("last_restock_date", to_date(col("last_restock_date")))
      .withColumn("modified_date", to_timestamp(col("modified_date")))
)

In [0]:
# PK FILTER (inventory_id must be valid)
df = df.filter(
    col("inventory_id").isNotNull() &
    (trim(col("inventory_id")) != "") &
    (col("inventory_id") != "0")
)

In [0]:
# OPTIONAL BUSINESS RULE (remove negative stock)
df = df.filter(col("stock_quantity") >= 0)


In [0]:
# DEDUPLICATION (Keep Latest Record)
w = Window.partitionBy("inventory_id").orderBy(
    col("modified_date").desc()
)

df = (
    df.withColumn("row_num", row_number().over(w))
      .filter(col("row_num") == 1)
      .drop("row_num")
)

In [0]:
# ADD INGESTION TIMESTAMP
df = df.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
# WRITE TO UNITY CATALOG
silver_table = "sales.silver.inventory"

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)

In [0]:
%sql
select * from sales.silver.inventory